In [ ]:
#训练模型y=w*x+b(简单线性回归)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
model=nn.Linear(1,1)
criterion=nn.MSELoss()
optimizer=optim.SGD(model.parameters(),lr=0.01)
x=torch.tensor([[1.0],[2.0],[3.0]])
y=torch.tensor([[2.0],[4.0],[6.0]])
for epoch in range(100):
    y_pred=model(x)
    loss=criterion(y_pred,y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch%20==0:
        print(f'Epoch {epoch},Loss:{loss.item():.4f}')
print(model)

In [ ]:
#在第一版训练结果的基础上修改训练轮数到1000，可以看到训练结果：最终损失降为0.0000（接近0），
#所以第二版（1000轮训练）的模型预测值和真实值几乎完全一致，该损失收敛情况稳定下降，没有震荡
#而且没有出现过拟合现象，几乎是完美拟合

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
model=nn.Linear(1,1)
criterion=nn.MSELoss()
optimizer=optim.SGD(model.parameters(),lr=0.01)
x=torch.tensor([[1.0],[2.0],[3.0]])
y=torch.tensor([[2.0],[4.0],[6.0]])
for epoch in range(1000):
    y_pred=model(x)
    loss=criterion(y_pred,y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch%20==0:
        print(f'Epoch {epoch},Loss:{loss.item():.4f}')

In [ ]:
#用梯度下降训练权重

In [ ]:
import torch
x=torch.tensor([[1.0],[2.0],[3.0]])
y=torch.tensor([[2.0],[4.0],[6.0]])
w=torch.tensor([[0.0]],requires_grad=True)
b=torch.tensor([[0.0]],requires_grad=True)
lr=0.01
for epoch in range(1000):#增大训练轮次，把损失降到最低，使预测值与真实值更加接近
    y_pred=x@w+b#利用前向传播
    loss=torch.mean((y-y_pred)**2)#定义损失
    loss.backward()#利用反向传播自动计算梯度
    with torch.no_grad():
        w-=lr*w.grad
        b-=lr*b.grad
        w.grad.zero_()
        b.grad.zero_()
    if epoch%40==0:
        print(f'Epoch{epoch:3d},w={w.item():.4f},b={b.item():.4f},loss={loss.item():.6f}')
print(f"\n最终：w={w.item():.4f},b={b.item():.4f}")

In [ ]:
#进阶练习

In [ ]:
#用梯度下降训练权重--房价预测

In [ ]:
#导入模型用到的库

In [1]:
import torch
import torch.nn as nn#(神经网络模块nn)
import torch.optim as optim#优化器模块
import matplotlib.pyplot as plt

In [2]:
#训练材料（数据导入）--房子的面积与卧室个数（用这两个数据来预测房价）

In [3]:
X=torch.tensor([
    [50.0,1.0],#第一个房子：50平米，1个卧室（以此类推接下来的房子数据
    [80.0,2.0],#第二个房子#注意用浮点数--pytorch默认用浮点数做梯度计算
    [100.0,2.0],
    [120.0,3.0],
    [150.0,3.0],
    [180.0,4.0],
    [200.0,5.0],
    [220.0,5.0],
    [240.0,6.0],
    [280.0,7.0],
])
y_true=torch.tensor([
    [150.0],#第一个房子的真实价格，后面以此类推
    [210.0],
    [270.0],
    [320.0],
    [380.0],
    [440.0],
    [500.0],
    [560.0],
    [620.0],
    [680.0],
])

In [4]:
#定义模型（设计房价预测算法）

In [5]:
class HousePriceModel(nn.Module):#（）里的内容表示继承，即这个类是从nn.Module里派生出来的
#nn.Module是pytorch提供的基类，包含了神经网络的基本功能
#模型继承后则有了自动求导、参数管理等功能
    def __init__(self):
        super().__init__()#调用父类（nn.Module）的初始化函数
        self.w1=nn.Parameter(torch.tensor([1.0]))#给模型（self指代）添加一个属性w1(权重)-面积价格系数，初始值为1.0
        self.w2=nn.Parameter(torch.tensor([20.0]))#w2(权重)--卧室单价系数--初始值为20.0
        self.b=nn.Parameter(torch.tensor([1.0]))#b--基础价格
        #被nn.Parameter包装后，pytorch会追踪它的梯度，自动求梯度
    def forward(self,x):#定义向前传播函数 #x是上面的数据X 面积与卧室个数
        return self.w1*x[:,0:1]+self.w2*x[:,1:2]+self.b
        #其中的：是取x的所有行 而0：1是表示从第0列到第1列（不包括1）--即取x第0列（面积）
        #所以x[:,0:1]是指取出面积这一列#结果：[[50],[80],...]
        #x[:,1:2]则是取出卧室个数

In [6]:
#创建模型实例

In [7]:
model=HousePriceModel()#创建了模型对象model 会调用__init__函数
#在model里w1 w2 b三个参数可以训练

In [8]:
print(model.w1)
print(model.w2)
print(model.b)#可打印出训练参数

Parameter containing:
tensor([1.], requires_grad=True)
Parameter containing:
tensor([20.], requires_grad=True)
Parameter containing:
tensor([1.], requires_grad=True)


In [9]:
#准备训练（训练函数）

In [10]:
criterion=nn.MSELoss()#创建均方误差损失函数#损失越小 模型预测越接近真实
#目标是让损失变小
optimizer=optim.Adam(model.parameters(),lr=0.005)
#使用SGD优化器 model.parameters()获取model所有可训练参数
#lr=learning rate学习率 不易太大太小 太小会训练过慢 太大可能会跳过最优点

In [11]:
#开始训练

In [12]:
for epoch in range(2000):#循环2000次从0到1999 一次即一个epoch
    y_pred=model(X)#调用model.forward(X) 输出X 输出预测价格
    loss=criterion(y_pred,y_true)#计算预测值和真实值的差距
    optimizer.zero_grad()#每轮开始前清零梯度#清空旧梯度
    loss.backward()#反向传播 计算本轮新梯度（每个参数的梯度）
    optimizer.step()#step()执行“梯度下降 更新参数”

In [13]:
#打印训练结果

In [14]:
print(f"w1={model.w1.item():.2f},w2={model.w2.item():.2f},b={model.b.item():.2f}")
#.item()把单元素张量转成python数字

w1=2.01,w2=20.74,b=3.97
